In [ ]:
import sys
sys.path.insert(0, 'ultralytics')
from ultralytics import settings
settings.update({"wandb": False})
import os
from ultralytics.models.yolo.model import YOLO

In [ ]:
best_hps = {'warmup_epochs': 3.6,
 'warmup_momentum': 0.77,
 'box': 8.13474,
 'cls': 0.51793,
 'dfl': 1.20629,
 'hsv_h': 0.05,
 'hsv_s': 0.38,
 'hsv_v': 0.25,
 'degrees': 30.0,
 'translate': 0.1,
 'scale': 0.4}

hps_set = {
'lr0': 0.001,
'lrf': 0.01,
"optimizer": "Adam",
 'momentum': 0.9,
 'weight_decay': 0.0005,
 'shear': 0.0,
 'perspective': 0.0,
 'flipud': 0.5,
 'fliplr': 0.5,
 'bgr': 0.0,
 'mosaic': 1.0,
 'mixup': 0.0,
 'copy_paste': 0.0,
'iou': 0.8
}
EPOCHS=1000
ES_PATIENCE=150
BATCH=64
WORKERS=2

MAIN_DS = 'datasets/baseline'
PRETRAINED_PATH = 'pretrained_weights/yolov8m-oiv7.pt'
DS_YAML_PATH = os.path.join(MAIN_DS, 'dataset.yaml')

# default
WORK_DIR = 'ultralytics/runs/detect'

In [ ]:
import torch
import gc
torch.cuda.empty_cache()
gc.collect()

# first training
model = YOLO(PRETRAINED_PATH)
train_results = model.train(data=DS_YAML_PATH, epochs=EPOCHS, batch=BATCH, workers=WORKERS, patience=ES_PATIENCE, 
                            **best_hps, **hps_set)
# will stop at the defined interrupt
del model
torch.cuda.empty_cache()
gc.collect()

In [ ]:
from utils import find_latest_checkpoint, find_latest_weights_dir

# progressive unfreezing

# unfreeze_at defines how many times to unfreeze at which epochs
# (trainer module was changed by us)
from ultralytics.engine.trainer import unfreeze_at

for i in range(len(unfreeze_at)):
    latest_checkpoint = os.path.abspath(find_latest_checkpoint(find_latest_weights_dir(WORK_DIR)))
    model = YOLO(latest_checkpoint)
    train_results = model.train(resume=True)
    del model
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
best_checkpoint = os.path.join(os.path.abspath(find_latest_weights_dir(WORK_DIR)), "best.pt")
yaml_test_filepath = os.path.join(DS_DIR, 'test_dataset.yaml')
model = YOLO(best_checkpoint)
model.val(data=yaml_test_filepath, conf=CONF_THRESH, iou=IOU, agnostic_nms=True)